# Anchor Similarity Filter — Calibration & Validation

This notebook calibrates and validates the FinBERT anchor similarity filter
before it is integrated into the main pipeline.

**Workflow:**
1. Load corpus and cached embeddings (no re-encoding)
2. Embed anchor phrases and compute per-article similarity scores
3. Decade stability analysis — does the filter behave consistently across eras?
4. Threshold calibration — where does the distribution suggest cutting?
5. Cross-validate against manual labels (paste from headline_inspection.ipynb)
6. Wave coverage check — does any survey wave go to zero after filtering?
7. Recommendation — final threshold and expected corpus reduction

**Prerequisite:** run the main pipeline notebook first so `article_embeddings`,
`news_df`, `all_results`, `SHILLER_FEATURES`, and the encoder are in memory.
Or set `STANDALONE = True` below to load from disk without the encoder.

## 1. Configuration

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from tqdm import tqdm
import warnings, re
warnings.filterwarnings("ignore")

# ── Paths (match main pipeline) ───────────────────────────────────────────────
DATA_DIR  = Path("./data")
CACHE_DIR = Path("./cache")

NEWS_PARQUETS   = [
    DATA_DIR / "wsj_headlines_1965_2014_zstd.parquet",
    DATA_DIR / "wsj_headlines_2015_2026_zstd.parquet",
]
NEWS_DATE_COL = "date"
NEWS_TEXT_COL = "headline"

# ── Anchor phrases ────────────────────────────────────────────────────────────
# These represent the semantic space of financially relevant content.
# Edit and re-run to test different anchor sets.
ANCHOR_PHRASES = [
    "corporate earnings quarterly results profit loss",
    "dividend payout shareholder return yield",
    "federal reserve interest rate monetary policy",
    "economic growth GDP inflation recession outlook",
    "stock market equity index trading shares",
    "merger acquisition takeover corporate deal",
    "employment jobs unemployment labor wages payroll",
    "oil energy commodity currency exchange rate dollar",
    "government spending fiscal deficit budget tax",
    "bank credit loan debt financial institution",
]

# ── Threshold grid to evaluate ────────────────────────────────────────────────
THRESHOLD_GRID = [0.60, 0.65, 0.70, 0.72, 0.75, 0.78, 0.80, 0.85]

# ── Walk-forward window (match main pipeline) ─────────────────────────────────
AGG_WINDOW_DAYS = 30

# ── Standalone mode (no encoder needed — uses cached embeddings only) ─────────
STANDALONE = False   # set True if running without the main pipeline kernel

print("Config loaded.")

## 2. Load corpus and cached embeddings

In [ ]:
# ── Load news corpus ──────────────────────────────────────────────────────────
try:
    _ = news_df
    print(f"Using news_df from main pipeline: {len(news_df):,} articles")
except NameError:
    frames = []
    for path in NEWS_PARQUETS:
        if not path.exists():
            print(f"WARNING: {path.name} not found — skipping.")
            continue
        df = pd.read_parquet(path)
        df = df.rename(columns={NEWS_DATE_COL: "date", NEWS_TEXT_COL: "text"})
        df = df[["date","text"]].dropna()
        df["date"] = pd.to_datetime(df["date"])
        frames.append(df)
        print(f"Loaded {len(df):,} from {path.name}")
    news_df = (pd.concat(frames, ignore_index=True)
                 .drop_duplicates(subset=["date","text"])
                 .sort_values("date")
                 .reset_index(drop=True))
    print(f"Combined: {len(news_df):,} articles")

news_df["year"]   = news_df["date"].dt.year
news_df["decade"] = (news_df["year"] // 10 * 10).astype(str) + "s"

# ── Load cached article embeddings ────────────────────────────────────────────
try:
    _ = article_embeddings
    print(f"Using article_embeddings from main pipeline: {article_embeddings.shape}")
except NameError:
    # Find the cache file matching the current corpus
    cache_key = (f"news_emb_n{len(news_df)}"
                 f"_{news_df['date'].min():%Y%m%d}"
                 f"_{news_df['date'].max():%Y%m%d}.npy")
    cache_path = CACHE_DIR / cache_key
    if cache_path.exists():
        print(f"Loading embeddings from {cache_path}...")
        article_embeddings = np.load(cache_path)
        print(f"Loaded: {article_embeddings.shape}")
    else:
        raise FileNotFoundError(
            f"Embedding cache not found at {cache_path}.\n"
            f"Run the main pipeline notebook first to generate embeddings."
        )

assert len(article_embeddings) == len(news_df),     f"Embedding count ({len(article_embeddings)}) != corpus size ({len(news_df)})"
print("\nCorpus and embeddings aligned.")

## 3. Embed anchor phrases and compute similarity

Cosine similarity between each article embedding and the anchor set.
The maximum similarity across all anchors is each article's relevance score.
This cell uses the cached article embeddings — no re-encoding.

In [ ]:
import torch

# ── Embed anchor phrases ───────────────────────────────────────────────────────
def embed_phrases(phrases, batch_size=32):
    """Embed a list of short phrases using the frozen FinBERT encoder."""
    if STANDALONE:
        raise RuntimeError("Encoder not available in STANDALONE mode. "
                           "Run main pipeline first.")
    all_emb = []
    model.eval()
    for i in range(0, len(phrases), batch_size):
        batch  = phrases[i:i+batch_size]
        inputs = tokenizer(batch, return_tensors="pt", padding=True,
                           truncation=True, max_length=64)
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
        with torch.no_grad():
            out = model(**inputs)
        all_emb.append(out.last_hidden_state[:, 0, :].cpu().numpy())
    return np.vstack(all_emb)


# ── Compute similarity ─────────────────────────────────────────────────────────
anchor_cache_path = CACHE_DIR / f"anchor_sim_n{len(news_df)}.npy"

if anchor_cache_path.exists():
    print(f"Loading cached anchor similarities from {anchor_cache_path}")
    anchor_sim = np.load(anchor_cache_path)
else:
    print("Embedding anchor phrases...")
    anchor_emb = embed_phrases(ANCHOR_PHRASES)

    print("Computing cosine similarity for all articles...")
    # Normalise both matrices for cosine similarity
    emb_norm    = article_embeddings / (
        np.linalg.norm(article_embeddings, axis=1, keepdims=True) + 1e-9
    )
    anchor_norm = anchor_emb / (
        np.linalg.norm(anchor_emb, axis=1, keepdims=True) + 1e-9
    )
    # Max similarity across all anchors per article — (n_articles,)
    # Process in batches to avoid OOM
    chunk = 50_000
    sims  = []
    for i in tqdm(range(0, len(emb_norm), chunk), desc="Similarity"):
        batch_sim = emb_norm[i:i+chunk] @ anchor_norm.T   # (chunk, n_anchors)
        sims.append(batch_sim.max(axis=1))
    anchor_sim = np.concatenate(sims)
    np.save(anchor_cache_path, anchor_sim)
    print(f"Cached to {anchor_cache_path}")

news_df["anchor_sim"]   = anchor_sim
news_df["best_anchor"]  = [ANCHOR_PHRASES[i]
                            for i in (article_embeddings /
                                      (np.linalg.norm(article_embeddings, axis=1,
                                                      keepdims=True)+1e-9)
                                      @ (embed_phrases(ANCHOR_PHRASES) /
                                         (np.linalg.norm(embed_phrases(ANCHOR_PHRASES),
                                                         axis=1, keepdims=True)+1e-9)).T
                                      ).argmax(axis=1)]     if not STANDALONE else ["unknown"] * len(news_df)

print(f"\nAnchor similarity stats:")
print(f"  mean:   {anchor_sim.mean():.4f}")
print(f"  median: {np.median(anchor_sim):.4f}")
print(f"  p10:    {np.percentile(anchor_sim, 10):.4f}")
print(f"  p25:    {np.percentile(anchor_sim, 25):.4f}")
print(f"  p75:    {np.percentile(anchor_sim, 75):.4f}")

## 4. Threshold calibration — full distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Anchor similarity distribution — full corpus", fontweight="bold")

ax = axes[0]
ax.hist(anchor_sim, bins=80, color="#2c5f8a", alpha=0.85, edgecolor="white")
ax.set_xlabel("Max cosine similarity to any anchor phrase")
ax.set_ylabel("Article count")
ax.set_title("Full corpus distribution")
colors_thresh = ["#e07b39","#e0b039","#4a9e6b","#9b59b6","#c0392b",
                 "#16a085","#2c5f8a","#8e44ad"]
for thresh, col in zip(THRESHOLD_GRID, colors_thresh):
    frac_drop = (anchor_sim < thresh).mean()
    ax.axvline(thresh, color=col, ls="--", lw=1.5,
               label=f">{thresh}: drop {frac_drop:.1%}")
ax.legend(fontsize=7.5); ax.grid(alpha=0.3)

ax = axes[1]
fracs_kept = [(anchor_sim >= t).mean() for t in THRESHOLD_GRID]
fracs_drop = [1 - f for f in fracs_kept]
ax.plot(THRESHOLD_GRID, fracs_kept, "o-", color="#2c5f8a", lw=2, label="Fraction kept")
ax.plot(THRESHOLD_GRID, fracs_drop, "s--", color="#c0392b", lw=2, label="Fraction dropped")
ax.set_xlabel("Threshold"); ax.set_ylabel("Fraction of corpus")
ax.set_title("Keep / drop rate by threshold")
ax.legend(); ax.grid(alpha=0.3); ax.set_ylim(0, 1)
for t, fk in zip(THRESHOLD_GRID, fracs_kept):
    ax.annotate(f"{fk:.1%}", (t, fk), textcoords="offset points",
                xytext=(0, 8), ha="center", fontsize=8)

plt.tight_layout()
plt.show()

print("\nThreshold summary:")
print(f"{'Threshold':>10} {'Keep':>10} {'Drop':>10}")
print("─" * 34)
for t in THRESHOLD_GRID:
    keep = (anchor_sim >= t).sum()
    drop = (anchor_sim <  t).sum()
    print(f"{t:>10.2f} {keep:>10,} ({keep/len(anchor_sim):.1%})  "
          f"{drop:>6,} ({drop/len(anchor_sim):.1%})")

## 5. Decade stability analysis

If FinBERT assigns systematically lower similarity scores to older headlines
(because they use different financial vocabulary), an aggressive threshold
would disproportionately filter early decades — damaging the historical
extrapolation period. This cell checks for that pattern.

In [ ]:
decades = sorted(news_df["decade"].unique())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Anchor similarity by decade", fontweight="bold")

# Panel 1: distribution per decade (violin)
ax = axes[0]
decade_data = [news_df[news_df["decade"]==d]["anchor_sim"].values for d in decades]
vp = ax.violinplot(decade_data, positions=range(len(decades)),
                   showmedians=True, showextrema=False)
for body in vp["bodies"]:
    body.set_facecolor("#2c5f8a"); body.set_alpha(0.6)
vp["cmedians"].set_color("#e07b39"); vp["cmedians"].set_linewidth(2)
ax.set_xticks(range(len(decades))); ax.set_xticklabels(decades)
ax.set_ylabel("Max anchor cosine similarity")
ax.set_title("Similarity distribution by decade")
for thresh in [0.70, 0.75]:
    ax.axhline(thresh, color="red", ls="--", lw=1, alpha=0.6, label=f"thresh={thresh}")
ax.legend(fontsize=8); ax.grid(alpha=0.3)

# Panel 2: fraction dropped per decade at each threshold
ax = axes[1]
for thresh, col in zip([0.70, 0.75, 0.80], ["#4a9e6b","#e07b39","#c0392b"]):
    drop_frac = [
        (news_df[news_df["decade"]==d]["anchor_sim"] < thresh).mean()
        for d in decades
    ]
    ax.plot(decades, drop_frac, "o-", color=col, lw=2, label=f"thresh={thresh}")
ax.set_ylabel("Fraction dropped")
ax.set_title("Drop rate by decade at candidate thresholds")
ax.legend(); ax.grid(alpha=0.3)
plt.setp(ax.get_xticklabels(), rotation=30, ha="right")

plt.tight_layout()
plt.show()

print("Median similarity by decade:")
for d in decades:
    med  = news_df[news_df["decade"]==d]["anchor_sim"].median()
    n    = (news_df["decade"]==d).sum()
    drop = (news_df[news_df["decade"]==d]["anchor_sim"] < 0.75).mean()
    print(f"  {d}  median={med:.4f}  n={n:>8,}  drop@0.75={drop:.1%}")

## 6. Cross-validation against manual labels

Paste your manual labels from `headline_inspection.ipynb` (section 6) here.
We match labels to similarity scores and compute precision/recall at each
threshold so you can pick the one that best matches your relevance judgement.

In [ ]:
# ── Reproduce the same sample from headline_inspection.ipynb ─────────────────
# Use the same LABEL_SEED and LABEL_N as you used there.
LABEL_SEED = 42
LABEL_N    = 100

label_sample = news_df.sample(LABEL_N, random_state=LABEL_SEED).reset_index(drop=True)

# ── Paste your r/i/u labels here ─────────────────────────────────────────────
# One label per line, same order as the inspection notebook printed.
# r = relevant, i = irrelevant, u = uncertain
MANUAL_LABELS = """
# paste labels here
"""

labels = [l.strip() for l in MANUAL_LABELS.strip().splitlines()
          if l.strip() and not l.strip().startswith("#")]

if len(labels) == LABEL_N:
    label_sample["label"]      = labels
    label_sample["anchor_sim"] = label_sample.index.map(
        lambda i: news_df.loc[news_df.index == label_sample.index[i],
                              "anchor_sim"].values[0]
        if label_sample.index[i] in news_df.index else np.nan
    )
    # Remap to anchor_sim using original index
    label_sample["anchor_sim"] = news_df.loc[label_sample.index, "anchor_sim"].values

    print(f"Label counts: r={labels.count('r')}  "
          f"i={labels.count('i')}  u={labels.count('u')}")
    print(f"\nSimilarity stats by label:")
    for lbl, name in [("r","relevant"), ("i","irrelevant"), ("u","uncertain")]:
        sub = label_sample[label_sample["label"]==lbl]["anchor_sim"]
        if len(sub):
            print(f"  {name:<12} n={len(sub):>3}  "
                  f"mean={sub.mean():.4f}  "
                  f"median={sub.median():.4f}  "
                  f"min={sub.min():.4f}")

    # Precision / recall at each threshold (treating r=1, i=0, ignoring u)
    labelled = label_sample[label_sample["label"].isin(["r","i"])].copy()
    labelled["true_relevant"] = (labelled["label"] == "r").astype(int)

    print(f"\nFilter performance (ignoring uncertain labels):")
    print(f"{'Threshold':>10} {'Precision':>11} {'Recall':>9} "
          f"{'F1':>7} {'Dropped':>9}")
    print("─" * 55)
    for thresh in THRESHOLD_GRID:
        pred_keep = labelled["anchor_sim"] >= thresh
        tp = (pred_keep &  labelled["true_relevant"].astype(bool)).sum()
        fp = (pred_keep & ~labelled["true_relevant"].astype(bool)).sum()
        fn = (~pred_keep & labelled["true_relevant"].astype(bool)).sum()
        prec = tp / (tp + fp) if (tp + fp) > 0 else 0
        rec  = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1   = 2*prec*rec/(prec+rec) if (prec+rec) > 0 else 0
        dropped_irr = (~pred_keep & ~labelled["true_relevant"].astype(bool)).sum()
        print(f"{thresh:>10.2f} {prec:>10.1%} {rec:>9.1%} "
              f"{f1:>7.3f} {dropped_irr:>5}/{(~labelled['true_relevant'].astype(bool)).sum()}")
else:
    print(f"Paste {LABEL_N} labels above (found {len(labels)}).")
    print("\nSimilarity distribution for this label sample:")
    sims = news_df.loc[label_sample.index, "anchor_sim"].values
    print(f"  mean={sims.mean():.4f}  median={np.median(sims):.4f}  "
          f"p10={np.percentile(sims,10):.4f}  p25={np.percentile(sims,25):.4f}")

## 7. Inspect what each threshold drops and keeps

The most important qualitative check — read through the kept and dropped
headlines at your candidate threshold before committing.

In [ ]:
INSPECT_THRESHOLD = 0.75   # change to your candidate threshold

kept    = news_df[news_df["anchor_sim"] >= INSPECT_THRESHOLD]
dropped = news_df[news_df["anchor_sim"] <  INSPECT_THRESHOLD]

print(f"Threshold = {INSPECT_THRESHOLD}")
print(f"Kept:    {len(kept):,} ({len(kept)/len(news_df):.1%})")
print(f"Dropped: {len(dropped):,} ({len(dropped)/len(news_df):.1%})")

print(f"\n── 25 random DROPPED headlines ──────────────────────────────────────")
for _, row in dropped.sample(25, random_state=1).iterrows():
    print(f"  {row['date'].strftime('%Y-%m-%d')}  sim={row['anchor_sim']:.3f}  {row['text']}")

print(f"\n── 15 random KEPT headlines (lowest similarity — borderline) ─────────")
borderline = kept.nsmallest(200, "anchor_sim").sample(15, random_state=1)
for _, row in borderline.iterrows():
    print(f"  {row['date'].strftime('%Y-%m-%d')}  sim={row['anchor_sim']:.3f}  {row['text']}")

## 8. Wave coverage check

Ensure no survey wave goes to zero or near-zero articles after filtering.
Waves with very few articles will have noisy embeddings.

In [ ]:
try:
    _ = all_results
    HAS_RESULTS = True
except NameError:
    HAS_RESULTS = False
    print("all_results not available — run main pipeline first for this cell.")

if HAS_RESULTS:
    MIN_ARTICLES_WARNING = 5   # flag waves with fewer than this many articles

    for series_name, res in all_results.items():
        waves = res["waves_df"]
        print(f"\n{'='*55}")
        print(f"  {series_name}  ({len(waves)} waves)")
        print(f"{'='*55}")
        print(f"{'Wave':<12} {'Raw':>6}", end="")
        for t in [0.70, 0.75, 0.80]:
            print(f"  {t:.2f}".rjust(10), end="")
        print()
        print("─" * 50)

        any_zero = {t: False for t in [0.70, 0.75, 0.80]}
        for _, wave in waves.iterrows():
            wd   = wave["wave_date"]
            mask = (
                (news_df["date"] >= wd - pd.Timedelta(days=AGG_WINDOW_DAYS)) &
                (news_df["date"] <  wd)
            )
            wave_df = news_df[mask]
            raw_n   = len(wave_df)
            print(f"{wd.strftime('%Y-%m'):>12} {raw_n:>6}", end="")
            for t in [0.70, 0.75, 0.80]:
                kept_n = (wave_df["anchor_sim"] >= t).sum()
                flag   = " ⚠" if kept_n < MIN_ARTICLES_WARNING else ""
                print(f"  {kept_n:>8}{flag}", end="")
                if kept_n == 0:
                    any_zero[t] = True
            print()

        print(f"\nZero-article waves:")
        for t, z in any_zero.items():
            print(f"  threshold={t}: {'YES ⚠' if z else 'none'}")

## 9. Anchor set sensitivity

Test whether a broader or narrower anchor set changes which headlines get
filtered. If results are stable the method is robust; if sensitive you
need to think more carefully about anchor choice.

In [ ]:
ANCHOR_SET_NARROW = [
    "corporate earnings quarterly profit loss revenue",
    "dividend payout shareholder yield",
    "stock market equity index fund shares",
    "merger acquisition takeover deal",
]

ANCHOR_SET_BROAD = ANCHOR_PHRASES + [
    "trade tariff import export international",
    "housing real estate mortgage construction",
    "healthcare pharmaceutical drug approval",
    "technology software patent innovation",
    "political election regulation government policy",
]

def compute_max_sim(emb_norm, phrases):
    anchor_emb = embed_phrases(phrases)
    anorm      = anchor_emb / (np.linalg.norm(anchor_emb, axis=1, keepdims=True)+1e-9)
    sims       = []
    for i in range(0, len(emb_norm), 50_000):
        sims.append((emb_norm[i:i+50_000] @ anorm.T).max(axis=1))
    return np.concatenate(sims)

if not STANDALONE:
    emb_norm = (article_embeddings /
                (np.linalg.norm(article_embeddings, axis=1, keepdims=True)+1e-9))

    print("Computing narrow anchor set similarity...")
    sim_narrow = compute_max_sim(emb_norm, ANCHOR_SET_NARROW)
    print("Computing broad anchor set similarity...")
    sim_broad  = compute_max_sim(emb_norm, ANCHOR_SET_BROAD)

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    fig.suptitle("Anchor set sensitivity", fontweight="bold")

    ax = axes[0]
    ax.hist(anchor_sim,  bins=60, alpha=0.5, label=f"Current ({len(ANCHOR_PHRASES)} anchors)",
            color="#2c5f8a", edgecolor="white")
    ax.hist(sim_narrow,  bins=60, alpha=0.5, label=f"Narrow ({len(ANCHOR_SET_NARROW)} anchors)",
            color="#e07b39", edgecolor="white")
    ax.hist(sim_broad,   bins=60, alpha=0.5, label=f"Broad ({len(ANCHOR_SET_BROAD)} anchors)",
            color="#4a9e6b", edgecolor="white")
    ax.set_xlabel("Max cosine similarity"); ax.set_ylabel("Count")
    ax.set_title("Distribution comparison"); ax.legend(fontsize=8); ax.grid(alpha=0.3)

    ax = axes[1]
    for thresh in [0.70, 0.75, 0.80]:
        drops = {
            "current": (anchor_sim < thresh).mean(),
            "narrow":  (sim_narrow  < thresh).mean(),
            "broad":   (sim_broad   < thresh).mean(),
        }
        ax.bar(
            [f"{k}\n@{thresh}" for k in drops],
            drops.values(),
            alpha=0.8, edgecolor="white"
        )
    ax.set_ylabel("Fraction dropped"); ax.set_title("Drop rate by anchor set & threshold")
    ax.grid(alpha=0.3, axis="y")

    plt.tight_layout()
    plt.show()

    corr_narrow = np.corrcoef(anchor_sim, sim_narrow)[0,1]
    corr_broad  = np.corrcoef(anchor_sim, sim_broad)[0,1]
    print(f"Pearson correlation with current set:")
    print(f"  Narrow: {corr_narrow:.4f}")
    print(f"  Broad:  {corr_broad:.4f}")
    print(f"\nHigh correlation (>0.95) = anchor set choice doesn't matter much.")
    print(f"Low correlation (<0.90) = worth experimenting with different anchors.")
else:
    print("STANDALONE mode — skip anchor sensitivity (encoder needed).")

## 10. Recommendation and next steps

In [ ]:
# ── Summary recommendation cell ──────────────────────────────────────────────
# Fill in CHOSEN_THRESHOLD after reviewing sections 4-9.
CHOSEN_THRESHOLD = 0.75   # update after review

kept_n  = (anchor_sim >= CHOSEN_THRESHOLD).sum()
drop_n  = (anchor_sim <  CHOSEN_THRESHOLD).sum()

print(f"{'='*55}")
print(f"  Recommended threshold: {CHOSEN_THRESHOLD}")
print(f"{'='*55}")
print(f"  Articles kept:   {kept_n:,} ({kept_n/len(news_df):.1%})")
print(f"  Articles dropped: {drop_n:,} ({drop_n/len(news_df):.1%})")
print()
print("Next steps:")
print("  1. Add ANCHOR_SIM_THRESHOLD to main pipeline config cell (cell 5)")
print("  2. Add ANCHOR_PHRASES list to config cell")
print("  3. In cache cell (cell 16): compute anchor_sim_all from cached embeddings")
print("  4. In aggregate_to_waves: add anchor_sim_all filter before sentiment filter")
print()
print("Implementation in aggregate_to_waves (add after NaN filter):")
print("""
    # Anchor similarity filter
    if anchor_sim_all is not None and anchor_threshold is not None:
        keep = anchor_sim_all[idx] >= anchor_threshold
        if keep.sum() == 0:
            keep[np.argmax(anchor_sim_all[idx])] = True  # keep best if all filtered
        emb, idx = emb[keep], idx[keep]
""")